In [1]:
import numpy as np
from astropy.table import Table
import os
import pandas as pd

In [2]:
def load_raw_df(path):
    tab = Table.read(path, format="fits", memmap=True)
    return tab.to_pandas()

def load_class_df(path):
    tab = Table.read(path, format="fits", memmap=True)
    return tab.to_pandas()

def get_zone_paths(raw_dir, class_dir, z):
    raw_path = os.path.join(raw_dir, f"zone_{z:02d}.fits.gz")
    cls_path = os.path.join(class_dir, f"zone_{z:02d}_classified.fits.gz")
    return raw_path, cls_path

def compute_r(df):
    denom = df["NDATA"] + df["NRAND"]
    # Avoid inf/NaN from zero denominators.
    df["r"] = np.where(denom != 0, (df["NDATA"] - df["NRAND"]) / denom, np.nan)
    return df

In [3]:
def ids_redshift(tracer, zone, z_min, z_max, raw_dir, class_dir):
    raw_path, cls_path = get_zone_paths(raw_dir, class_dir, zone)
    raw_df = load_raw_df(raw_path)

    for col in raw_df.columns:
        raw_df[col] = raw_df[col].apply(lambda x: x.decode() if isinstance(x, bytes) else x)

    raw_fil = raw_df[raw_df['TRACERTYPE'] == tracer]

    tr_interval = raw_fil[(raw_fil['Z'] >= z_min) & (raw_fil['Z'] <= z_max)]
    tr_ids = tr_interval['TARGETID'].tolist()

    return tr_ids

def class_df(tracer, zone, raw_dir, class_dir, data):

    raw_path, cls_path = get_zone_paths(raw_dir, class_dir, zone)
    cls_df = load_raw_df(cls_path)

    for col in cls_df.columns:
        cls_df[col] = cls_df[col].apply(lambda x: x.decode() if isinstance(x, bytes) else x)

    if data == 'DATA':
        cls_df = cls_df[cls_df['ISDATA'] == True]
    elif data == 'RAND':
        cls_df = cls_df[cls_df['ISDATA'] == False]
    else:
        raise ValueError("Data must be 'DATA' or 'RAND'")
    
    tracer_map = {
        'BGS_ANY_DATA': 'BGS_ANY',
        'BGS_ANY_RAND': 'BGS_ANY',
        'LRG_DATA': 'LRG',
        'LRG_RAND': 'LRG',
        'ELG_DATA': 'ELG',
        'ELG_RAND': 'ELG',
        'QSO_DATA': 'QSO',
        'QSO_RAND': 'QSO'
    }

    tracer_norm = tracer_map.get(tracer, tracer)

    cls_fil = cls_df[cls_df['TRACERTYPE'] == tracer_norm]

    return cls_fil

def assign_classification(cls_df):

    conditions = [
        (cls_df['r'] >= -1) & (cls_df['r'] <= -0.25),
        (cls_df['r'] > -0.25) & (cls_df['r'] <= 0.25),
        (cls_df['r'] > 0.25) & (cls_df['r'] <= 0.65),
        (cls_df['r'] > 0.65) & (cls_df['r'] <= 1)
    ]

    choices = ['void', 'sheet', 'filament', 'knot']

    cls_df['classification'] = np.select(conditions, choices, default='undefined')

    return cls_df


def classification_fraction_randiter_stats(cls_df):

    total_per_iter = cls_df.groupby('RANDITER').size()
    counts = cls_df.groupby(['RANDITER', 'classification']).size()

    fractions = (counts / total_per_iter).reset_index(name='fraction')

    frac_pivot = fractions.pivot(
        index='RANDITER',
        columns='classification',
        values='fraction'
    ).fillna(0)

    return frac_pivot


def count_fraction(zones, data, raw_dir, class_dir, tracer=None, dz=0.1):

    tracer_redshift_ranges = {
        'BGS_ANY_DATA': (0.0, 0.6),
        'BGS_ANY_RAND': (0.0, 0.6),
        'LRG_DATA': (0.0, 1.1),
        'LRG_RAND': (0.0, 1.1),
        'ELG_DATA': (0.0, 1.6),
        'ELG_RAND': (0.0, 1.6),
        'QSO_DATA': (0.0, 3.5),
        'QSO_RAND': (0.0, 3.5),
    }

    if data == 'DATA':
        all_tracers = ['BGS_ANY_DATA', 'LRG_DATA', 'ELG_DATA', 'QSO_DATA']
    elif data == 'RAND':
        all_tracers = ['BGS_ANY_RAND', 'LRG_RAND', 'ELG_RAND', 'QSO_RAND']
    else:
        raise ValueError("Data must be 'DATA' or 'RAND'")

    if tracer is not None:
        tracers = [tracer]
    else:
        tracers = all_tracers

    tracer_results = {}

    invalid_tracers = [tr for tr in tracers if tr not in tracer_redshift_ranges]
    if invalid_tracers:
        raise ValueError(
            f"Unknown tracer(s): {invalid_tracers}. Valid options: {sorted(tracer_redshift_ranges)}"
        )

    for tr in tracers:
        print(f"\nTracer: {tr}")

        z_start, z_end = tracer_redshift_ranges[tr]
        redshifts = np.round(np.arange(z_start, z_end + dz, dz), 2)

        final_rows = []

        for i in range(len(redshifts) - 1):
            z_min = redshifts[i]
            z_max_interval = redshifts[i+1]
            print(f"  Redshift bin: {z_min:.1f}-{z_max_interval:.1f}")

            zone_fractions = []

            for z in range(zones):

                tr_ids = ids_redshift(
                    tracer=tr,
                    zone=z,
                    z_min=z_min,
                    z_max=z_max_interval,
                    raw_dir=raw_dir,
                    class_dir=class_dir
                )

                cls_df = class_df(
                    tracer=tr,
                    zone=z,
                    raw_dir=raw_dir,
                    class_dir=class_dir,
                    data=data
                )

                cls_df = cls_df[cls_df['TARGETID'].isin(tr_ids)]

                if len(cls_df) == 0:
                    continue

                compute_r(cls_df)
                cls_df = assign_classification(cls_df)

                counts = cls_df['classification'].value_counts(normalize=True)

                zone_row = counts.to_dict()
                zone_row['zone'] = z
                zone_fractions.append(zone_row)

            if len(zone_fractions) == 0:
                continue

            zone_df = pd.DataFrame(zone_fractions).fillna(0)

            mean_vals = zone_df.drop(columns='zone').mean() * 100
            std_vals  = zone_df.drop(columns='zone').std() * 100

            for cls in mean_vals.index:
                final_rows.append({
                    "tracer": tr,
                    "ZMIN": z_min,
                    "ZMAX": z_max_interval,
                    "classification": cls,
                    "mean_percentage": mean_vals[cls],
                    "std_percentage": std_vals[cls]
                })

        tracer_results[tr] = pd.DataFrame(final_rows)

    return tracer_results


In [4]:
base_dir = "/Users/soguevaram/Desktop/Research/DESI/data"
raw_dir = os.path.join(base_dir, "edr/raw")
class_dir = os.path.join(base_dir, "edr/classification")

In [5]:
results_rand_bgs = count_fraction(
               zones=20,
               data='RAND',
               raw_dir=raw_dir,
               class_dir=class_dir,
               tracer='BGS_ANY_RAND',
               dz=0.6)


Tracer: BGS_ANY_RAND
  Redshift bin: 0.0-0.6


In [6]:
results_data_bgs = count_fraction(
               zones=20,
               data='DATA',
               raw_dir=raw_dir,
               class_dir=class_dir,
               tracer='BGS_ANY_DATA',
               dz=0.6)


Tracer: BGS_ANY_DATA
  Redshift bin: 0.0-0.6


In [7]:
results_rand_lrg = count_fraction(
               zones=20,
               data='RAND',
               raw_dir=raw_dir,
               class_dir=class_dir,
               tracer='LRG_RAND',
               dz=1.1)


Tracer: LRG_RAND
  Redshift bin: 0.0-1.1


In [8]:
results_data_lrg = count_fraction(
               zones=20,
               data='DATA',
               raw_dir=raw_dir,
               class_dir=class_dir,
               tracer='LRG_DATA',
               dz=1.1)


Tracer: LRG_DATA
  Redshift bin: 0.0-1.1


In [9]:
results_rand_elg = count_fraction(
               zones=20,
               data='RAND',
               raw_dir=raw_dir,
               class_dir=class_dir,
               tracer='ELG_RAND',
               dz=1.6)


Tracer: ELG_RAND
  Redshift bin: 0.0-1.6


In [10]:
results_data_elg = count_fraction(
               zones=20,
               data='DATA',
               raw_dir=raw_dir,
               class_dir=class_dir,
               tracer='ELG_DATA',
               dz=1.6)


Tracer: ELG_DATA
  Redshift bin: 0.0-1.6


In [11]:
results_rand_qso = count_fraction(
               zones=20,
               data='RAND',
               raw_dir=raw_dir,
               class_dir=class_dir,
               tracer='QSO_RAND',
               dz=3.5)


Tracer: QSO_RAND
  Redshift bin: 0.0-3.5


In [12]:
results_data_qso = count_fraction(
               zones=20,
               data='DATA',
               raw_dir=raw_dir,
               class_dir=class_dir,
               tracer='QSO_DATA',
               dz=3.5)


Tracer: QSO_DATA
  Redshift bin: 0.0-3.5


In [13]:
print(results_rand_bgs)
print(results_rand_lrg)
print(results_rand_elg)
print(results_rand_qso)

{'BGS_ANY_RAND':          tracer  ZMIN  ZMAX classification  mean_percentage  std_percentage
0  BGS_ANY_RAND   0.0   0.6           void        59.411131        1.150954
1  BGS_ANY_RAND   0.0   0.6          sheet        29.741538        1.136603
2  BGS_ANY_RAND   0.0   0.6       filament         8.980180        0.322734
3  BGS_ANY_RAND   0.0   0.6           knot         1.867151        0.154242}
{'LRG_RAND':      tracer  ZMIN  ZMAX classification  mean_percentage  std_percentage
0  LRG_RAND   0.0   1.1           void        47.409524        1.076524
1  LRG_RAND   0.0   1.1          sheet        41.584215        1.135981
2  LRG_RAND   0.0   1.1       filament         9.918159        0.306680
3  LRG_RAND   0.0   1.1           knot         1.088103        0.086738}
{'ELG_RAND':      tracer  ZMIN  ZMAX classification  mean_percentage  std_percentage
0  ELG_RAND   0.0   1.6          sheet        50.347021        0.733617
1  ELG_RAND   0.0   1.6           void        36.137884        0.719283

In [14]:
print(results_data_bgs)
print(results_data_lrg)
print(results_data_elg)
print(results_data_qso)

{'BGS_ANY_DATA':          tracer  ZMIN  ZMAX classification  mean_percentage  std_percentage
0  BGS_ANY_DATA   0.0   0.6          sheet        34.163952        1.465675
1  BGS_ANY_DATA   0.0   0.6       filament        30.704858        0.782467
2  BGS_ANY_DATA   0.0   0.6           knot        18.795771        1.707444
3  BGS_ANY_DATA   0.0   0.6           void        16.335419        0.462093}
{'LRG_DATA':      tracer  ZMIN  ZMAX classification  mean_percentage  std_percentage
0  LRG_DATA   0.0   1.1          sheet        44.248374        1.160409
1  LRG_DATA   0.0   1.1       filament        27.181212        0.719745
2  LRG_DATA   0.0   1.1           void        19.717921        0.411875
3  LRG_DATA   0.0   1.1           knot         8.852494        0.923667}
{'ELG_DATA':      tracer  ZMIN  ZMAX classification  mean_percentage  std_percentage
0  ELG_DATA   0.0   1.6          sheet        52.725239        0.727130
1  ELG_DATA   0.0   1.6       filament        23.911049        0.449193